# L7b: Parameters and Sensitivity Analysis
In this lecture, we'll explore a simple example of parameter estimation, and will introduce local and global sensitivity analysis. Sensitivity analysis will help us understand how the model output changes with respect to changes in the model parameters, and will give us some insight into the indetifiability of the model parameters. The key ideas of this lecture are:
* __Least squares minimization__ is an optimization problem used to estimate model parameters. It involves minimizing the squared difference between experimental data and model predictions. Various optimization algorithms can solve this problem. Today, we'll use [simulated annealing](https://en.wikipedia.org/wiki/Simulated_annealing) to solve a sample problem.
* __Local sensitivity analysis__ is a method to understand how the model output changes with respect to changes in the model parameters _locally_, i.e., in the infetisimally small region of some $\theta_{\circ}$, one at a time. It involves calculating the partial derivative of the model output with respect to each parameter. This values give us insight into the identifiability of the model parameters, and a nice linkage to the design of experiments.
* __Global sensitivity analysis__ is a method to understand how the model output changes with respect to large changes in the model parameters. It involves sampling the response of the model to changes in the parameters over a large range of values, where paramerers can be changing jointly. This method is useful to understand the correlation between the model parameters, and to identify the most influential parameters on the model output.

The notes for today were be taken from several sources:
1. [Gadkar KG, Varner J, Doyle FJ 3rd. Model identification of signal transduction networks from data using a state regulator problem. IEE Syst Biol. 2005 Mar;2(1):17-30. doi: 10.1049/sb:20045029. PMID: 17091579.](https://digital-library.theiet.org/doi/abs/10.1049/sb%3A20045029)
2. [Hu CY, Varner JD, Lucks JB. Generating Effective Models and Parameters for RNA Genetic Circuits. ACS Synth Biol. 2015 Aug 21;4(8):914-26. doi: 10.1021/acssynbio.5b00077. Epub 2015 Jul 2. PMID: 26046393.](https://pubmed.ncbi.nlm.nih.gov/26046393/)

## Example: Parameter Estimation of mAb Production in Batch Culture
In this example, we'll consider a simple model of monoclonal antibody (mAb) production in a (potentially) variable volume culture. The model is given by the following set of ordinary differential equations (ODEs):
$$
\begin{align*}
\frac{dS}{dt} &= D\left(S_{1} - S\right) - \frac{1}{Y_{X/S}^{\star}}\mu X - \frac{1}{Y_{P/S}}q_{p} X\\
\frac{dP}{dt} &= D\left(P_{1}- P\right) + q_{p}X\\
\frac{dX}{dt} &= \left(\mu - k_{d}\right)X - DX \\
\frac{dV}{dt} &= F\left(t\right)
\end{align*}
$$
where $S$ is the substrate concentration, $P$ is the product concentration, $X$ is the biomass concentration, $V$ is the volume of the culture, and $D \equiv F/V$ is the diluation rate. The $S_{1}$ and $P_{1}$ are the substrate and product concentrations in the feed, respectively. The $Y_{X/S}^{*}$ and $Y_{P/S}$ are yield coefficients (experimental parameters) for biomass and product, respectively. The $\mu$ is the specific growth rate of the biomass, $q_{p}$ is the specific product formation rate, and $k_{d}$ is the death rate of the biomass. The $F\left(t\right)$ is the input volumetric flow rate function, which can be specified by the user.

The specific growth rate of the biomass, $\mu$, is given by the [Monod equation](https://en.wikipedia.org/wiki/Monod_equation):
$$
\begin{equation*}
	\mu = \mu_{g}^{max}\left(\frac{S}{K_{g} + S}\right)
\end{equation*}
$$ 
and we assume the [Luedeking and Piret model](https://pubmed.ncbi.nlm.nih.gov/26038085/) for product formation:
$$
\begin{equation*}
	q_{p} = \alpha~\mu+\beta
\end{equation*}
$$
The input volumetric flow rate function $F(t)$ is specified by the user; different functions (or parameters within typical functions such as an exponential feed function) strongly influence the performance of the culture. The cellmass, substrate, product and volume balances are coupled nonlinear differential equations which can be solved numerically using packages [such as the `Sundials.jl` package](https://github.com/SciML/Sundials.jl) which is a wrapper around the [Sundials C library](https://computing.llnl.gov/projects/sundials).

### Batch Culture
In a batch culture, the dilution rate $D$ is zero, and the volume of the culture is constant. The ODEs for the batch culture are given by:
$$
\begin{align*}
\frac{dS}{dt} &= - \frac{1}{Y_{X/S}^{\star}}\mu X - \frac{1}{Y_{P/S}}q_{p} X\\
\frac{dP}{dt} &= q_{p}X\\
\frac{dX}{dt} &= \left(\mu - k_{d}\right)X \\
\end{align*}
$$
